In [ ]:
Content:
1. Frameworks
2. Design multi-step workflows or tool integrations?
3. LLM Limitation

{(1, 2, 3, 4): 2}

# 1. Frameworks

In [ ]:
# Orchestration frameworks
    * LangChain for tool integrations, RAG pipelines, chaining, and workflow orchestration
    * CrewAI for multi-agent collaboration workflows such as research, validation, and summarization agents
    * OpenAI APIs for lightweight orchestration, function/tool calling, and lower-latency production systems
    



# 2. Design multi-step workflows or tool integrations?

In [ ]:
Two Approaches:
    1. Agent = LLM picks tools & order (flexible) ← Most common
    2. SequentialChain = Fixed step order (simpler)


In [ ]:
# First Approach: LLM picks tools & order (flexible)
    User Question
        ↓
    LLM thinks: "I need to search customer AND check inventory"
        ↓
    Calls Tool 1: SearchCustomer("123")
        ↓ (Gets: "Customer 123: Status=Active, Product=Widget")
    Calls Tool 2: CheckInventory("Widget")
        ↓ (Gets: "Widget: In Stock (50 units)")
        ↓
    LLM synthesizes: "Yes, customer 123 can get their Widget (50 available)"

# Code
from langchain.llms import OpenAI
from langchain.tools import Tool
from langchain.agents import initialize_agent, AgentType
from langchain.chains import LLMChain, SequentialChain
from langchain.prompts import PromptTemplate

# Step 1: Define Tools (what the LLM can use)
def search_customer_db(customer_id: str) -> str:
    """Get customer info from database"""
    return f"Customer {customer_id}: Status=Active, Tickets=5"

def check_inventory(product: str) -> str:
    """Check if product is in stock"""
    return f"{product}: In Stock (50 units)"

tools = [
    Tool(
        name="SearchCustomer",
        func=search_customer_db,
        description="Search customer database by ID"
    ),
    Tool(
        name="CheckInventory",
        func=check_inventory,
        description="Check product inventory"
    ),
]

# Step 2: Create Agent (orchestrates tool use)
llm = OpenAI(temperature=0)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Step 3: Run workflow (LLM decides which tools to use, in what order)
response = agent.run(
    "Check if customer 123 has stock for their product"
)
# LLM automatically:
# 1. Calls SearchCustomer(123)
# 2. Calls CheckInventory(product_name)
# 3. Synthesizes answer

In [ ]:
# Second approach: Sequential Chain (Simpler, Fixed Order)
    - If you do not need LLM to decide order, use SequentialChain

from langchain.chains import SequentialChain, LLMChain
from langchain.prompts import PromptTemplate

# Define steps that run in order
step1 = LLMChain(
    llm=llm,
    prompt=PromptTemplate(
        input_variables=["customer_id"],
        template="Find customer {customer_id} details"
    ),
    output_key="customer_info"
)

step2 = LLMChain(
    llm=llm,
    prompt=PromptTemplate(
        input_variables=["customer_info"],
        template="Based on {customer_info}, recommend a product"
    ),
    output_key="recommendation"
)

# Chain them
workflow = SequentialChain(
    chains=[step1, step2],
    input_variables=["customer_id"],
    output_variables=["recommendation"],
    verbose=True
)

result = workflow({"customer_id": "123"})


In [ ]:
3. QnA

Q1: How to reduce LATENCY in production system?
    a. Caching (Easiest)
        - if a customer calls or a generic apis calls, we save the result in Redis. So, If any other user ask the same query, we retrun the result from redis.

    b. Streaming
        - Do not wait for full response, show words as they appear. Even though full answer takes 8s, it FEELS instant.
    c. Smart Models selection
        - Simple tasks (summarize, categorize) → use gpt-3.5-turbo (50ms, cheap)
        - Complex tasks (debug, design) → use gpt-4 (300ms, expensive)
        - Do not use slow expensive model for everything
    d. Parallel Calls
        - Instead of: Get customer → wait → Get orders → wait → Get tickets
        - Do: Get customer AND orders AND tickets simultaneously

Q2: How to reduce HALLUCINATION?
    Issue: User: "How many open tickets do I have?"
            - LLM: "You have 5 tickets" (Actually you have 8, LLM guessed wrong)
            - User acts on wrong information → disaster

        a. RAG (Retrieval-Augmented Generation) - Most effective
            - Do not ask LLM to remember facts
            - Instead: Get real data from database → show to LLM → LLM synthesizes
            - Example:
                 - Database: "Customer ABC: 8 open tickets (as of now)"
                 - You feed to LLM: ↑ this document
                 - LLM responds: "According to system, you have 8 tickets"

        b. Tool Use (For calculations/data)
            - Do not ask LLM "What's 523 × 47?" (it guesses)
            - Instead: Tell LLM Use calculator tool → we do the math → give answer

        c. Fine-tuning (If you have specific domain)
            - LLM trained on internet (general knowledge)
            - Your company has specific procedures, terminology, products
            - Fine-tune LLM on YOUR data (50-100 examples)
            - LLM learns: "When asked about X, always say Y"


Q3: How to reduce COST?
    a. Smart Model Routing 
        - For Simple tasks like (summarize, extract, categorize): use chat models/ models without thinking
        - For Complex tasks like (debuging, designing): use models with thinking capabilities.
        - Claude models: Haiku (cheap for simple tasks) -> Sonet -> Opus (mostly for complex tasks).

    b. Output Limits (Don't ask for novel when you need summary)
        - In your prompts (user/ system injected prompts): Add what exactly the goal is or how detailed answer is required.

    c. Caching
    d. Batch Processing: Do Batch Processing if possible as per business usecase.
            - OpenAI and Anthophic usually provide upto 50% off in Batch processing.
    

# 3. QnA Project

In [ ]:

**1. System Design – A Complex Automation I Built**

I built a RAG pipeline over 5,000+ PDFs to handle customer support queries for air cargo operations. The stack was Python + LangChain + ChromaDB for vector storage, with OpenAI (GPT-4o) as the reasoning layer, all deployed on AWS Lambda behind an API Gateway.

The core problem was this: support agents were manually searching through IATA cargo documentation and airline-specific PDFs to answer queries — a process that could take 15–20 minutes per ticket. The automation ingested all documents, chunked and embedded them into ChromaDB, and exposed a retrieval + synthesis API. When a query came in, the system retrieved the top-K relevant chunks, injected them into a structured prompt, and returned a grounded answer — not a guess.

Success was measured on three fronts: resolution time (down ~40%), hallucination rate (near-zero, since answers were document-grounded), and support team adoption (tracked via internal usage logs). The last one honestly mattered the most to me — a system no one uses is just a demo.

---

**2. Error Handling – Graceful Failures in Multi-Step Workflows**

This is something I've had to solve specifically in high-throughput cargo messaging pipelines where we processed 10K+ shipments daily. The short answer: I never let two dependent steps sit in the same synchronous call chain if either of them can be slow or flaky.

In practice, I design around a queue-backed retry pattern. Each step writes its output to a durable store (SQS or a database staging table) before triggering the next step. If the next step fails, it doesn't lose the data — it just retries from the queue with exponential backoff. For API rate limits specifically, I add a dead-letter queue and an alerting hook so nothing silently disappears.

For AI-specific steps, I also keep timeouts intentionally short and wrap the LLM call in a fallback: if it times out or errors, the record gets flagged and queued for a retry rather than dropped. The key principle I follow is — every step should be idempotent and every failure should be recoverable, not terminal.

---

**3. Async Communication – Documenting for Non-Technical Stakeholders**

When I led the IATA ONE Record migration at WiseTech, the technical complexity was significant — transitioning legacy Cargo-IMP messaging (FWB/FHL) to a new API-based standard across multiple airlines. But the harder challenge was getting airline operations teams and product managers to understand what was changing and why.

What worked for me was separating the "what it does" layer from the "how it works" layer entirely. I wrote a one-page business flow doc with a simple visual: data comes in here → gets validated here → goes out there. No code, no jargon. Then separately, I maintained a technical spec for the engineering team. The operations teams only ever saw the first one.

I also recorded short Loom walkthroughs for async review — especially useful since some stakeholders were across different time zones. That combination (written doc + short video) reduced back-and-forth questions significantly. People could refer back to it without pinging me.

---

**4. System Review – Re-architecting the Lead Capture Pipeline**

The problem is a classic synchronous bottleneck. The webhook waits for OpenAI, OpenAI takes 30+ seconds, HubSpot times out. The fix is to decouple the steps so no single timeout can break the chain.

Here's how I'd re-architect it:

1. **Webhook → Queue**: The webhook immediately writes the raw lead payload to a durable queue (SQS or similar) and returns a 200 to the form. Job done — the lead is safe.

2. **Queue → AI Scoring Worker**: A background worker picks up the lead, sends it to OpenAI for scoring, and waits however long it takes. There's no timeout pressure here because it's async. If OpenAI fails, the message stays in the queue and retries with backoff.

3. **Worker → HubSpot**: Once the score comes back, the worker writes to HubSpot. If HubSpot is also flaky, same pattern — retry from queue, dead-letter after N attempts, alert on failure.

The end result: the webhook is always fast, no leads are ever lost, and each step fails independently without cascading. The tradeoff is that HubSpot might not see the lead for 30–60 seconds instead of instantly — which for a lead scoring use case is almost always completely fine.

---

Looking forward to the next steps. Happy to jump on a call if anything here needs more context.

Best,
Sourav Aggarwal

# 3. LLM Limitation

In [ ]:
1. Hallucination
    - Problem: LLM confidently generates wrong information.
    - Solution: RAG, Tool Calling, Web search, Lower temperature

2. LLMs sometimes forget or ignore information present in the middle of long context.
    - LLM mainly focus on start and end information.
    - Solution: Add verification layer, Gurdrails, like if it has followed all the instructions correctly.

3. Vague/ Unclear Input
    - User input statement can be solve in thousands of way, so a LLM might do something which you might not want.
    - Solution: Provide a detailed context, Prompt Engineering.

4. No Real time knowledge
    - Solution: RAG, funtion calling, web search
        
5. Security: Prompt Injection
    - Example: User manipulates instructions, show credentials.
    - Solution: Add Guardrails, Instructions, and specification and better to add a validation layer.

# 4. Reduce Token Usage

In [ ]:
1. Disconnect MCP server if not required.
    1. One server alone 18K token per message.
    2. instead of using MCP, use CLI as much as you can. for example:  google calender cli
2. use /clear between unrelated tasks
3. use Plan model, instead of implementing it in first go.
    - Add this to your CLAUDE.md: Do not make any changes until you have 95% confidence in what you need to build. Ask me follow-up questions until you reach that confidence."
4. Claude commands to check
    1. /context 
    2. /cost
    3. /statusline



# 5. Important file writing Instructions

In [ ]:
1. CLAUDE.md
    1. Place it in your project root. Claude auto-reads it at the start of every chat as system context. <200 lines
    2.  Include: your tech stack, coding conventions, build commands, the 95% confidence rule.
    3. Treat it like an index. Route to where more data lives. This is a mindset shift.